In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
import sys

# Suppress TensorFlow warnings
warnings.filterwarnings('ignore', category=FutureWarning)
tf.get_logger().setLevel('ERROR')

# Define the file path
file_path = '/content/drive/MyDrive/MScDS&AI/ANN/student-mat.csv'

# --- Step 1: Read the CSV file ---
# The metadata indicates the delimiter is ';'
# Removed try/except block as requested
df = pd.read_csv(file_path, delimiter=';')
print(f"Successfully loaded '{file_path}'\n")

# --- Step 2: Display columns and type ---
print("--- Column Info (Initial) ---")
df.info()


Successfully loaded '/content/drive/MyDrive/MScDS&AI/ANN/student-mat.csv'

--- Column Info (Initial) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null 

In [3]:

# --- Step 3: Introduce a new column 'High_G3' ---
# G3 is the final grade
df['High_G3'] = np.where(df['G3'] > 10, 'High', 'Moderate')

# --- Step 4: Print the distribution of 'High_G3' ---
print("--- Distribution of New Target Column 'High_G3' ---")
print(df['High_G3'].value_counts())


# --- Step 5: Remove G3 column ---
df = df.drop('G3', axis=1)

# --- Step 6: Prepare X (features) and y (target) ---
y = df['High_G3']
X = df.drop('High_G3', axis=1)

# --- Step 7: Convert categorical features to one-hot encoding ---
# We will also convert binary 'yes'/'no' columns
# Find object-type columns to encode
categorical_cols = X.select_dtypes(include=['object']).columns

# Use pd.get_dummies for one-hot encoding
X_processed = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"--- Preprocessing ---")
print(f"Original feature count: {len(X.columns)}")
print(f"Feature count after one-hot encoding: {len(X_processed.columns)}")


--- Distribution of New Target Column 'High_G3' ---
High_G3
High        209
Moderate    186
Name: count, dtype: int64
--- Preprocessing ---
Original feature count: 32
Feature count after one-hot encoding: 41


In [4]:

# --- Step 8: StandardScaler features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_processed)

# --- Step 9: Perform train-test split (80-20) ---
# We need to convert y to numerical labels (0 for Moderate, 1 for High)
y_numerical = y.map({'Moderate': 0, 'High': 1})

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y_numerical,
    test_size=0.2,
    random_state=42,
    stratify=y_numerical  # Helps keep distribution in train/test
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")



Training samples: 316
Testing samples: 79


In [7]:
# --- Step 10: Fit ML Perceptron using Sequential() ---

# Get the number of input features after processing
n_features = X_train.shape[1]

# Initialize the model
model = Sequential()

# Add Input layer and first hidden layer (32 neurons)
model.add(Dense(32, activation='relu', input_shape=(n_features,)))

# Add second hidden layer (16 neurons)
model.add(Dense(16, activation='relu'))

# Add Output layer (1 neuron, sigmoid for binary classification)
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='mean_squared_error',metrics=['accuracy'])

print("--- Model Summary ---")
model.summary()



--- Model Summary ---


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 32)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,889 (7.38 KB)

 Trainable params: 1,889 (7.38 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:

# Train (fit) the model
print("--- Training Model ---")
history = model.fit(
    X_train,
    y_train,
    epochs=50,  # Number of passes through the data
    batch_size=16, # Number of samples per update
    validation_data=(X_test, y_test),
    verbose=1 # Show progress
)
print("Model training complete.\n")


--- Training Model ---
Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.4907 - loss: 0.2961 - val_accuracy: 0.5316 - val_loss: 0.2844
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6108 - loss: 0.2178 - val_accuracy: 0.5949 - val_loss: 0.2551
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6647 - loss: 0.2027 - val_accuracy: 0.5949 - val_loss: 0.2329
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7617 - loss: 0.1752 - val_accuracy: 0.6456 - val_loss: 0.2138
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7667 - loss: 0.1646 - val_accuracy: 0.6709 - val_loss: 0.1956
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8333 - loss: 0.1464 - val_accuracy: 0.6835 - val_loss: 0.1807
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8390 - loss: 0.1283 - val_accuracy: 0.7215 - val_loss: 0.1664
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8918 - loss: 0.0995 - val_accu

In [9]:


# --- Step 11: Evaluate the model ---
print("--- Model Evaluation on Test Set ---")

# Get predictions (probabilities)
y_pred_probs = model.predict(X_test)

# Convert probabilities to class labels (0 or 1)
y_pred = (y_pred_probs > 0.5).astype(int)

# Get target names
target_names = ['Moderate (0)', 'High (1)']

# 1. Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}\n")

# 2. Confusion Matrix
print("--- Confusion Matrix ---")
# Rows are Actual, Columns are Predicted
#      Pred_Mod  Pred_High
# Act_Mod   TN        FP
# Act_High  FN        TP
cm = confusion_matrix(y_test, y_pred)
print(cm)
print("\n")

# 3. Classification Report (Precision, Recall, F1-score, Support)
print("--- Classification Report ---")
report = classification_report(y_test, y_pred, target_names=target_names)
print(report)



--- Model Evaluation on Test Set ---
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
Accuracy: 0.8861

--- Confusion Matrix ---
[[31  6]
 [ 3 39]]


--- Classification Report ---
              precision    recall  f1-score   support

Moderate (0)       0.91      0.84      0.87        37
    High (1)       0.87      0.93      0.90        42

    accuracy                           0.89        79
   macro avg       0.89      0.88      0.88        79
weighted avg       0.89      0.89      0.89        79

